# Text-to-Image Generation with Stable Diffusion and Diffusers

Generate images from text prompts using the [Diffusers](https://huggingface.co/docs/diffusers) library.
Covers two model tiers:
- **SD 1.5-based** — `dreamlike-art/dreamlike-diffusion-1.0` (lighter, fits on small GPUs)
- **SDXL** — `stabilityai/stable-diffusion-xl-base-1.0` (higher quality, needs ~8GB VRAM)

> **GPU recommended.** Runs on CPU but is very slow.

## Installation

In [ ]:
!pip install -q diffusers transformers accelerate

## Imports & Device Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import DiffusionPipeline, StableDiffusionXLPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device.upper()}  |  dtype: {dtype}")

---
## Helper — Display Images

In [ ]:
def show_images(images, prompts=None):
    """Display one or more PIL images side-by-side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 7))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, prompts or [""] * n):
        ax.imshow(img)
        ax.axis("off")
        if title:
            ax.set_title(title[:60], fontsize=9, wrap=True)
    plt.tight_layout()
    plt.show()

---
## Part 1 — SD 1.5-based: Dreamlike Diffusion

`DiffusionPipeline.from_pretrained()` auto-detects the correct pipeline class from the model card —
no need to import `StableDiffusionPipeline` explicitly.

In [ ]:
pipe = DiffusionPipeline.from_pretrained(
    "dreamlike-art/dreamlike-diffusion-1.0",
    torch_dtype=dtype,
    use_safetensors=True,
)

# Memory optimisations
if device == "cuda":
    pipe.enable_attention_slicing()   # reduces peak VRAM on small GPUs

pipe = pipe.to(device)

### Basic Generation

In [ ]:
prompt1 = (
    "dreamlikeart, a grungy woman with rainbow hair, travelling between dimensions, "
    "dynamic pose, happy, soft eyes and narrow chin, extreme bokeh, dainty figure, "
    "long hair straight down, torn kawaii shirt and baggy jeans"
)

# torch.Generator for reproducible results
generator = torch.Generator(device=device).manual_seed(42)

result = pipe(prompt1, generator=generator)
show_images(result.images, [prompt1])

In [ ]:
prompt2 = (
    "A girl sitting on a chair accompanied by her tiger, "
    "cinematic lighting, golden iris color palette, dramatic atmosphere"
)

generator = torch.Generator(device=device).manual_seed(0)
result = pipe(prompt2, generator=generator)
show_images(result.images, [prompt2])

---
## Part 2 — Working with Generation Parameters

Key parameters for controlling generation quality and style:

| Parameter | What it does |
|---|---|
| `num_inference_steps` | More steps → more detail, slower (default 50) |
| `guidance_scale` | How closely to follow the prompt (default 7.5; higher = more literal) |
| `height` / `width` | Output resolution in pixels (must be multiples of 8) |
| `num_images_per_prompt` | Number of images to generate in one call |
| `negative_prompt` | What to avoid in the generated image |
| `generator` | Seed for reproducibility |

In [ ]:
def generate_images(pipe, prompt, negative_prompt=None, **kwargs):
    """Generate and display images with optional parameters."""
    result = pipe(
        prompt,
        negative_prompt=negative_prompt,
        **kwargs,
    )
    show_images(result.images)
    return result.images

In [ ]:
festival_prompt = (
    "dreamlike, beautiful woman playing Holi festival of colors, "
    "draped in traditional Indian attire, throwing vibrant powder, golden hour lighting"
)

# Default parameters
generate_images(pipe, festival_prompt)

In [ ]:
# More inference steps → sharper, more detailed result
generate_images(pipe, festival_prompt, num_inference_steps=75)

In [ ]:
# guidance_scale: how strictly to follow the prompt
# Lower (~5) = more creative, Higher (~12) = more literal
generate_images(pipe, festival_prompt, num_inference_steps=50, guidance_scale=12.0)

In [ ]:
# Custom resolution — dimensions must be multiples of 8
generate_images(pipe, festival_prompt, num_inference_steps=50, width=512, height=768)

In [ ]:
# Generate multiple images at once
generate_images(pipe, festival_prompt, num_inference_steps=50, num_images_per_prompt=2)

In [ ]:
# Negative prompt — steer away from unwanted artefacts
generate_images(
    pipe,
    festival_prompt,
    negative_prompt="ugly, distorted, low quality, blurry, watermark, bad anatomy",
    num_inference_steps=50,
    num_images_per_prompt=2,
)

### Reproducibility with Seeds

Same seed + same prompt always produces the same image.

In [ ]:
seed_prompt = "A majestic dragon flying over a misty mountain, fantasy art, highly detailed"

images = []
for seed in [42, 42, 99]:  # seed 42 twice — should produce identical images
    g = torch.Generator(device=device).manual_seed(seed)
    images.append(pipe(seed_prompt, generator=g, num_inference_steps=30).images[0])

show_images(images, [f"seed={s}" for s in [42, 42, 99]])

---
## Part 3 — Stable Diffusion XL (SDXL)

SDXL produces significantly higher-quality outputs at 1024×1024 natively.  
Requires ~8GB VRAM. Uses `enable_model_cpu_offload()` to fit on 8–10GB GPUs.

In [ ]:
pipe_xl = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=dtype,
    use_safetensors=True,
)

# CPU offload: moves layers to CPU when not in use — fits SDXL on ~8GB VRAM
if device == "cuda":
    pipe_xl.enable_model_cpu_offload()
else:
    pipe_xl = pipe_xl.to(device)

In [ ]:
xl_prompt = (
    "A breathtaking aerial view of a futuristic city at sunset, neon lights reflecting "
    "on rain-soaked streets, cyberpunk aesthetic, ultra-detailed, cinematic lighting"
)

generator = torch.Generator(device=device).manual_seed(42)

result = pipe_xl(
    prompt=xl_prompt,
    negative_prompt="blurry, low quality, watermark, ugly",
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
)

show_images(result.images, [xl_prompt])